In [3]:
import pandas as pd
import s3fs

MINIO_ENDPOINT = "http://169.58.96.176:9000"
MINIO_KEY = "admin"
MINIO_SECRET = "adminGrupo3"
BUCKET_NAME = "programacion-ciencia-datos-g3"

storage_opts = {
    "key": MINIO_KEY,
    "secret": MINIO_SECRET,
    "client_kwargs": {"endpoint_url": MINIO_ENDPOINT}
}

print("🔄 Iniciando el proceso del punto 5.4: Integración y Validación...")

# 1. Carga de los datasets transformados desde la Capa Silver
df_trans = pd.read_parquet(
    f"s3://{BUCKET_NAME}/silver/train_transaction_transformed.parquet", 
    storage_options=storage_opts
)
df_id = pd.read_parquet(
    f"s3://{BUCKET_NAME}/silver/train_identity_transformed.parquet", 
    storage_options=storage_opts
)

print(f"   • Dataset de Transacciones cargado: {df_trans.shape[0]:,} filas, {df_trans.shape[1]} columnas")
print(f"   • Dataset de Identidad cargado:    {df_id.shape[0]:,} filas, {df_id.shape[1]} columnas")

# ==============================================================================
# 5.4 INTEGRACIÓN Y VALIDACIÓN
# ==============================================================================

print("\n--- A. COMBINACIÓN DE DATASETS (MERGE) ---")
# Realizamos un Left Join utilizando 'TransactionID' como llave primaria relacional
df_master = pd.merge(df_trans, df_id, on="TransactionID", how="left")

print(f"✨ Dimensiones de la Tabla Maestra Integrada: {df_master.shape[0]:,} filas y {df_master.shape[1]} columnas")

print("\n--- B. VALIDACIÓN DE REGISTROS E INTEGRIDAD ---")
# 1. Validación de cardinalidad y conservación de filas
assert df_master.shape[0] == df_trans.shape[0], "Error crítico: El número de registros transaccionales varió tras el cruce."
print("✔️ [Validación Aprobada] El total de transacciones (590,540) se mantiene intacto.")

# 2. Conteo de valores nulos estructurales tras la integración con identidad
nulos_identidad_cruzada = df_master['DeviceType'].isnull().sum()
print(f"✔️ [Validación Aprobada] Transacciones sin huella de identidad asociada: {nulos_identidad_cruzada:,} (esperado debido a que no todas las operaciones poseen registro de dispositivo).")

print("\n--- C. CONSISTENCIA DE RESULTADOS ---")
# Verificación de consistencia en la etiqueta objetivo y montos
fraudes_totales = df_master['isFraud'].sum()
monto_global_pen = df_master['monto_pen'].sum()
print(f"• Total de fraudes confirmados en la tabla maestra: {fraudes_totales:,}")
print(f"• Importe monetario global acumulado (PEN): S/ {monto_global_pen:,.2f}")

# Muestra final integrada para el informe
print("\n📋 Muestra de la Tabla Maestra Integrada:")
display(df_master[['TransactionID', 'isFraud', 'fecha_hora', 'card4', 'monto_pen', 'DeviceType', 'familia_os']].head(5))

🔄 Iniciando el proceso del punto 5.4: Integración y Validación...
   • Dataset de Transacciones cargado: 590,540 filas, 22 columnas
   • Dataset de Identidad cargado:    144,233 filas, 10 columnas

--- A. COMBINACIÓN DE DATASETS (MERGE) ---
✨ Dimensiones de la Tabla Maestra Integrada: 590,540 filas y 31 columnas

--- B. VALIDACIÓN DE REGISTROS E INTEGRIDAD ---
✔️ [Validación Aprobada] El total de transacciones (590,540) se mantiene intacto.
✔️ [Validación Aprobada] Transacciones sin huella de identidad asociada: 446,307 (esperado debido a que no todas las operaciones poseen registro de dispositivo).

--- C. CONSISTENCIA DE RESULTADOS ---
• Total de fraudes confirmados en la tabla maestra: 20,663
• Importe monetario global acumulado (PEN): S/ 299,020,672.00

📋 Muestra de la Tabla Maestra Integrada:


,TransactionID,isFraud,fecha_hora,card4,monto_pen,DeviceType,familia_os
0,2987000,0,2026-01-02 00:00:00,discover,256.880005,NaN,NaN
1,2987001,0,2026-01-02 00:00:01,mastercard,108.750000,NaN,NaN
2,2987002,0,2026-01-02 00:01:09,visa,221.250000,NaN,NaN
3,2987003,0,2026-01-02 00:01:39,mastercard,187.500000,NaN,NaN
4,2987004,0,2026-01-02 00:01:46,mastercard,187.500000,mobile,Android
